# 03 · 특징 하나로 어디까지 예측할 수 있는가

02의 특징 28개를 하나씩 사용해 단일 지표 모델을 만든다. 각 특징에 선형 로지스틱과 spline 로지스틱을 적합하므로 후보는 총 56개다. 검증 AUROC로 각 계열의 특징 하나를 선택하고, 선택을 고정한 뒤 시험 성능을 계산한다. 이후 조합 모델과 비교할 기준을 만드는 단계다.

**흐름:** 특징·분할 읽기 → 모델 정의와 작은 예시 → 56개 후보 학습 → 검증으로 선택 → 예측 → 시험 지표와 산모 신뢰구간 → 곡선·저장

**입력:** `data/results/features.csv`, `protocol.json`  
**출력:** 같은 폴더의 후보 순위, 선택 기록, 검증·시험 예측, 시험 지표와 실행 상태

`03-요약된-파일/03-단일-지표.ipynb`의 후보군, 학습 설정, 선택 규칙과 bootstrap을 유지한다. 원래 `study.py`에 있던 필요한 계산을 이 노트북 안에서 확인할 수 있도록 풀었다. 입력 대상과 산모 분할은 `04`의 00 → 01 → 02 결과를 사용한다.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import average_precision_score, roc_auc_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import SplineTransformer, StandardScaler

RESULTS = Path("data/results")
# Match the original model and mother-bootstrap settings.
MODEL_SEED = 42
BOOTSTRAP_SEED = 42
N_BOOTSTRAP = 2000

### 출력에 남기는 오류 정보

오류는 고정된 분류와 건수로만 요약한다. `other_error`의 상세 원인은 내부 오류 파일에서 확인한다. 건수는 오류 내역 수이며, 같은 기록이 여러 번 포함될 수 있다.

In [ ]:
def issue_category(reason):
    # Match known phrases, but return ONLY fixed labels: never echo an exception.
    # Example: "FileNotFoundError: /private/DEMO.json" -> "missing_input".
    text = str(reason)
    groups = {
        "missing_input": ("FileNotFoundError", "No such file", "[Errno 2]"),
        "missing_dependency": ("No module named", "cannot import name", "ImportError"),
        "fhr_missing": ("FHR 결측률",),
        "invalid_wave": ("FHR과 TOCO", "측정 간격", "FHR은 0 이상", "신호 길이"),
        "invalid_bbox": ("Bbox",),
        "invalid_id": ("ID가", "산모 ID", "산모 목록", "행 정보 불일치", "구간 정보 불일치"),
        "invalid_labels": ("라벨이 올바르지", "라벨 수", "라벨·점수"),
        "no_usable_data": ("구간이 없습니다", "비었", "남은 산모가 없습니다"),
        "split_unavailable": ("분할", "양성·음성", "정상·비정상", "least populated class", "test_size"),
        "checkpoint_mismatch": ("체크포인트", "CNN 입력·설정"),
        "io_error": ("PermissionError", "Permission denied", "No space left", "OSError"),
        "invalid_value": ("ValueError", "TypeError", "유한", "could not convert", "ufunc"),
    }
    for category, phrases in groups.items():
        if any(phrase in text for phrase in phrases):
            return category
    return "other_error"

In [ ]:
def show_issue_counts(reasons):
    # Count issue entries, not unique people; one record may have several entries.
    categories = pd.Series([issue_category(reason) for reason in reasons], dtype="string")
    counts = categories.value_counts().rename_axis("category").to_frame("count")
    print("제외·오류 내역 수 =", len(categories))
    # Full messages, IDs, paths, and jobs stay in the internal CSV files.
    with pd.option_context("display.max_rows", None, "display.max_columns", None):
        display(counts)

## 1. 같은 특징과 산모 분할을 이어받는다

한 행은 02의 특징 추출에 성공한 구간이다. `target`은 해당 구간의 전문가 이상 라벨, `split`은 01에서 정한 산모별 배정이다. 다시 분할하지 않는다. 특징 순서는 검증 점수가 같은 후보 사이의 선택 순서이기도 하다.

In [ ]:
# Keep the Cat28 order from 02; validation-AUROC ties use this order.
CAT28 = [
    "fhr_mean", "fhr_min", "fhr_max", "fhr_sd", "stv", "brady_frac", "tachy_frac",
    "n_decel", "decel_max_depth", "decel_time_frac", "n_accel",
    "toco_mean", "toco_max", "toco_sd", "n_contractions",
    "ft_corr0", "ft_corr_min", "ft_corr_min_lag_s",
    "figo_baseline", "figo_baseline_var",
    "n_early_decel", "n_late_decel", "n_variable_decel",
    "n_severe_decel", "n_prolonged_decel", "hist_width", "hist_median", "hist_mode",
]
# 28 features × 2 families = 56 candidate models.
FAMILIES = ("linear", "spline")

In [ ]:
# One row in features.csv is one retained fetus segment, including its
# target and the mother-level split assigned by 01.
feature_table = pd.DataFrame()
protocol = {}
input_error = None

try:
    # IDs are strings; "00DEMO527183" is a fictional example with leading zeros.
    id_types = {key: str for key in ("row_id", "record_id", "mother_id", "site")}
    feature_table = pd.read_csv(RESULTS / "features.csv", dtype=id_types)
    # protocol.json says whether 01 could create a usable split.
    protocol = json.loads((RESULTS / "protocol.json").read_text(encoding="utf-8"))
except (OSError, ValueError) as error:
    input_error = str(error)

In [ ]:
ready = False
training = pd.DataFrame()
validation = pd.DataFrame()
testing = pd.DataFrame()

if input_error is None:
    try:
        if feature_table.empty or not protocol.get("split_ready", False):
            raise ValueError("특징 표가 비었거나 01의 산모 분할이 준비되지 않았습니다")
        # Example: missing "hist_width" means its linear and spline
        # candidates cannot be formed, so stop before training.
        feature_table[CAT28]  # Every candidate must have its input column.
        if not feature_table["row_id"].is_unique:
            raise ValueError("특징 표의 구간 ID가 중복됩니다")
        # Example: all rows for mother "DEMO622612" must share one split,
        # even when they come from repeated records or twins.
        if feature_table.groupby("mother_id")["split"].nunique().max() > 1:
            raise ValueError("같은 산모가 여러 분할에 배정되어 있습니다")

        # Use the saved assignments; do not draw a new split here.
        training = feature_table.loc[feature_table["split"].eq("train")].copy()
        validation = feature_table.loc[feature_table["split"].eq("val")].copy()
        testing = feature_table.loc[feature_table["split"].eq("test")].copy()

        # After feature exclusions, all three splits still need labels 0 and 1.
        # Example: a test split containing only normal windows cannot yield AUROC.
        for part in (training, validation, testing):
            if set(part["target"].unique()) != {0, 1}:
                raise ValueError("train/val/test 각각 정상·비정상 라벨이 필요합니다")
        ready = True
    except (KeyError, ValueError) as error:
        input_error = str(error)

In [ ]:
print("ready =", ready)
if ready:
    # Count mothers once per split; repeated recordings may contain many windows.
    # Example: 2 records from one mother count as 1 mother, while
    # abnormal_fraction counts abnormal windows / all windows.
    split_summary = feature_table.groupby("split").agg(
        mothers=("mother_id", "nunique"),
        windows=("row_id", "size"),
        abnormal_fraction=("target", "mean"),
    )
    display(split_summary)
else:
    print('단일 지표 실험 미산출:', '[details omitted from notebook output]')

## 2. 특징 하나를 직선 또는 곡선으로 읽는다

선형 로지스틱은 특징값이 커질수록 점수가 한 방향으로 변한다. spline 로지스틱은 특징 하나를 여러 기저값으로 바꿔 U자형 같은 관계도 표현할 수 있다.

$$p_L(z)=\sigma(\beta_0+\beta z),\qquad
p_S(z)=\sigma\left(\beta_0+\sum_h\beta_h B_h(z)\right).$$

$\sigma$는 값을 0–1 확률로 바꾸는 시그모이드다. spline은 학습 범위의 등간격 knot 5개와 3차 다항식으로 만든다. 학습 범위 밖에서는 경계의 기저값을 유지한다. 상수 특징은 0열 하나로 바꿔 절편만 학습한다.

In [ ]:
class SplineBasis(TransformerMixin, BaseEstimator):
    def fit(self, values, y=None):
        # Example: [140, 140, 140] has range 0 and needs no spline basis.
        # np.ptp is max - min per column. [110, 140, 170] has range 60.
        self.keep_ = np.ptp(values, axis=0) > 0
        self.spline_ = None
        if self.keep_.any():
            # Example: training range 110–170 -> 5 uniform knots
            # [110, 125, 140, 155, 170], fitted on training data only.
            self.spline_ = SplineTransformer(
                n_knots=5, degree=3, knots="uniform",
                extrapolation="constant", include_bias=False,
            )
            self.spline_.fit(values[:, self.keep_])
        return self

    def transform(self, values):
        if self.spline_ is None:
            # Example: a constant feature for 4 windows becomes shape (4, 1).
            return np.zeros((len(values), 1))
        # A varying single feature becomes 6 spline-basis columns.
        # "constant" extrapolation reuses the end basis beyond train range.
        return self.spline_.transform(values[:, self.keep_])

결측 대체 → 선형 입력 또는 spline 기저 → 표준화 → 로지스틱 회귀를 하나의 pipeline으로 묶는다. 중앙값, knot 범위, 평균·표준편차와 모델 계수는 모두 학습 집합에서만 정한다. 전부 결측인 열은 0으로 대체한다. 정규화 강도 `C=1`, 최대 반복 2,000회, seed 42를 유지한다.

In [ ]:
def make_single_model(family):
    # Example: train values [110, 130, NaN, 170] use median 130 for imputation.
    steps = [SimpleImputer(strategy="median", keep_empty_features=True)]
    # Example: a train column that is entirely missing becomes 0,
    # then the spline arm uses its constant-feature fallback.
    if family == "spline":
        steps.append(SplineBasis())

    # Scaling is fitted after the optional spline expansion.
    steps.append(StandardScaler())
    # C=1 controls coefficient regularization; max_iter limits fitting.
    steps.append(LogisticRegression(C=1, max_iter=2000, random_state=MODEL_SEED))
    return make_pipeline(*steps)

In [ ]:
# A small synthetic example; these rows are not used in the real experiment.
example_values = pd.DataFrame({"fhr_mean": [110.0, 130.0, np.nan, 170.0]})
example_targets = np.array([0, 0, 1, 1])
example_model = make_single_model("spline")
example_model.fit(example_values, example_targets)

print("input shape =", example_values.shape)
print("imputation median =", example_model.named_steps["simpleimputer"].statistics_)
# One varying spline input -> 6 columns after imputation and scaling.
print("transformed shape =", example_model[:-1].transform(example_values).shape)
print("predicted probabilities =", example_model.predict_proba(example_values)[:, 1])

# Example: all-constant input uses one zero column instead of six spline columns.
constant_basis = SplineBasis().fit_transform(np.full((4, 1), 140.0))
print("constant basis =", constant_basis.tolist())

## 3. 56개 후보를 학습하고 검증 점수를 모은다

`fit_candidate`는 특징 하나로 학습한 모델과 검증 점수 한 행을 반환한다. `[[feature]]`는 학습 입력을 `(구간 수, 1)`인 표로 유지한다. 검증 데이터는 예측과 점수 계산에만 사용하며 시험 점수는 아직 계산하지 않는다.

In [ ]:
def fit_candidate(family, feature):
    model = make_single_model(family)
    # Double brackets keep a two-dimensional input.
    # Example: 8 training rows and one feature -> shape (8, 1).
    model.fit(training[[feature]], training["target"])

    # predict_proba gives [P(normal), P(abnormal)] per row; column 1 is the score.
    # Example: [0.8, 0.2] -> score 0.2 for the abnormal class.
    # Validation uses the fitted train pipeline; it does not refit it.
    scores = model.predict_proba(validation[[feature]])[:, 1]
    result = dict(
        family=family,
        feature=feature,
        # AUROC ranks positives above negatives; AP rewards precision
        # at each newly recovered positive. Only AUROC selects later.
        val_AUROC=roc_auc_score(validation["target"], scores),
        val_AP=average_precision_score(validation["target"], scores),
    )
    return model, result

In [ ]:
models = {}
ranking_rows = []
issues = []

if ready:
    for family in FAMILIES:
        for feature in CAT28:
            try:
                model, result = fit_candidate(family, feature)
                # Keep the fitted model and its score under the same
                # key, e.g. ("spline", "hist_width").
                models[(family, feature)] = model
                ranking_rows.append(result)
            except Exception as error:
                # Preserve the source's candidate-level failure handling.
                # One failed fit is recorded; the remaining candidates still run.
                issues.append(dict(stage="fit", job=f"{family}:{feature}", reason=str(error)))
        # Report once per family, never once per segment or fit.
        print(family, "completed candidates =", sum(row["family"] == family for row in ranking_rows))

In [ ]:
# Example: one row might be [linear, fhr_mean, AUROC=0.80, AP=0.60].
# The stored row order is family order, then Cat28 feature order.
ranking = pd.DataFrame(
    ranking_rows, columns=["family", "feature", "val_AUROC", "val_AP"],
)
print("successful candidates =", len(ranking), "/", len(FAMILIES) * len(CAT28))
print("failed candidates =", len(issues))
display(ranking.sort_values("val_AUROC", ascending=False, kind="stable").head(12))

## 4. 검증 AUROC로 계열별 하나를 고정한다

각 계열의 28개 후보가 모두 성공해야 선택한다. 실패한 후보를 빼고 최선을 고르면 후보군 자체가 달라지기 때문이다. 검증 AUROC가 같으면 Cat28 순서에서 먼저 나온 특징을 고른다. AP는 함께 보고하지만 선택에는 사용하지 않는다.

선택된 모델은 학습 집합에서 적합한 상태로 고정한다. 학습+검증으로 다시 적합하거나 두 모델의 시험 점수를 보고 선택을 바꾸지 않는다.

In [ ]:
def choose_candidate(candidates):
    # Example: 27 successful features are not a complete family of 28;
    # do not choose the best among only those 27.
    if len(candidates) != len(CAT28):
        raise ValueError("28개 후보가 모두 완료되지 않아 선택을 보류합니다")

    # Stable sorting preserves Cat28 order when validation AUROC is tied.
    # Example: [fhr_mean=0.80, fhr_min=0.80] selects fhr_mean.
    ordered = candidates.sort_values("val_AUROC", ascending=False, kind="stable")
    return ordered.iloc[0]

In [ ]:
# Example: equal scores preserve the input feature order.
example_ranking = pd.DataFrame({
    "feature": ["fhr_mean", "fhr_min"],
    "val_AUROC": [0.80, 0.80],
})
example_order = example_ranking.sort_values("val_AUROC", ascending=False, kind="stable")
print("tied example selects =", example_order.iloc[0]["feature"])

In [ ]:
selected_models = {}
selections = {}

if ready:
    for family in FAMILIES:
        # Linear and spline each compete only with their own 28 features.
        candidates = ranking.loc[ranking["family"].eq(family)]
        try:
            best = choose_candidate(candidates)
        except ValueError as error:
            issues.append(dict(stage="selection", job=family, reason=str(error)))
            continue

        feature = best["feature"]
        arm = f"single_{family}"
        # Reuse this train-fitted model on test; validation only selected it.
        selected_models[arm] = models[(family, feature)]
        selections[arm] = dict(
            feature=feature, selection="validation_AUROC", val_AUROC=float(best["val_AUROC"]),
        )

print("selected models =", selections)

## 5. 선택된 모델의 검증·시험 예측을 만든다

구간 ID·산모 ID·라벨·분할에 예측 확률 `score`를 붙인다. 나중에 모델끼리 같은 구간을 비교할 수 있도록 원래 ID와 `signal_index`를 남긴다. 아직 경보 임계값을 정하지 않았으므로 `threshold`는 결측이다.

In [ ]:
# These columns identify exactly which window, mother, label and split
# produced each probability in the saved prediction table.
PREDICTION_COLUMNS = [
    "row_id", "record_id", "mother_id", "site", "target",
    "split", "record_all_normal", "signal_index",
]


def make_predictions(frame, scores):
    # Example: row_id="DEMO221841_0:0", score=0.7 means P(abnormal)=0.7 for that row.
    prediction = frame[PREDICTION_COLUMNS].copy()
    prediction["score"] = np.asarray(scores, dtype=float)
    # No alarm threshold is chosen in this experiment.
    prediction["threshold"] = np.nan
    if prediction["row_id"].duplicated().any() or not np.isfinite(prediction["score"]).all():
        raise ValueError("예측 키가 중복되거나 점수가 유한하지 않습니다")
    return prediction

In [ ]:
# Empty schemas are saved too, so an unavailable arm cannot leave old predictions.
prediction_tables = {}
for family in FAMILIES:
    prediction_tables[f"single_{family}"] = pd.DataFrame(
        columns=PREDICTION_COLUMNS + ["score", "threshold"],
    )

if selected_models:
    # Example: 10 validation + 5 test rows -> 15 predictions per selected arm.
    evaluation_pool = pd.concat([validation, testing])
    for arm, model in selected_models.items():
        feature = selections[arm]["feature"]
        # Each arm uses the one feature chosen by validation AUROC.
        scores = model.predict_proba(evaluation_pool[[feature]])[:, 1]
        prediction_tables[arm] = make_predictions(evaluation_pool, scores)

In [ ]:
for arm, prediction in prediction_tables.items():
    print(arm, "prediction rows =", len(prediction))
    # Both val and test are stored; the next section evaluates only test rows.

## 6. 시험 성능을 산모 단위 신뢰구간과 함께 계산한다

### 6.1. 점수와 산모 묶음을 준비한다

AP는 양성을 더 찾아갈 때의 정밀도를 합친 순위 지표이고, AUROC는 양성이 음성보다 높은 점수를 받는 정도를 나타낸다. Brier는 확률과 0·1 라벨의 제곱 오차 평균이다. AP·AUROC는 클수록, Brier는 작을수록 좋다.

동일 산모의 모든 구간에 같은 bootstrap 가중치를 준다. 산모를 두 번 뽑으면 그 산모의 모든 구간을 두 번 센다. 신뢰구간은 학습·선택을 다시 수행하지 않고 고정한 모델의 시험 예측을 재표집해서 구한다.

In [ ]:
def prepare_metrics(prediction):
    labels = prediction["target"].to_numpy(dtype=int)
    scores = prediction["score"].to_numpy(dtype=float)
    if not len(prediction) or set(labels) - {0, 1} or not np.isfinite(scores).all():
        raise ValueError("평가 라벨·점수가 올바르지 않거나 평가 집합이 비었습니다")

    # Example: mother IDs [A, A, B, B] -> mother_index [0, 0, 1, 1].
    mothers, mother_index = np.unique(prediction["mother_id"], return_inverse=True)
    # Example: scores [0.2, 0.9, 0.5] -> descending row order [1, 2, 0].
    order = np.argsort(-scores, kind="stable")

    # Count tied scores together. Sorted scores [0.9, 0.9, 0.3] -> ends [1, 2].
    # The last index closes the final score group. AP and AUROC treat
    # equal scores as a single threshold, not arbitrary row order.
    group_ends = np.r_[np.flatnonzero(np.diff(scores[order])), len(prediction) - 1]
    return dict(
        labels=labels, scores=scores, mothers=mothers,
        mother_index=mother_index, order=order, group_ends=group_ends,
    )

In [ ]:
def weighted_metrics(data, counts=None):
    if counts is None:
        # Point estimate: count every test mother exactly once.
        counts = np.ones(len(data["mothers"]))
    # Example: mother counts [2, 0] -> row weights [2, 2, 0, 0] for [A,A,B,B].
    weights = counts[data["mother_index"]]
    labels, scores = data["labels"], data["scores"]
    positive = np.sum(weights * labels)
    negative = np.sum(weights * (1 - labels))
    # A bootstrap draw containing only one label has no AUROC; skip it.
    if not positive or not negative:
        return None

    order, ends = data["order"], data["group_ends"]
    # At each distinct score, count all windows at or above it.
    # Example: labels [1, 0, 1] -> cumulative TP [0, 1, 1, 2].
    true_positive = np.r_[0, np.cumsum((weights * labels)[order])[ends]]
    false_positive = np.r_[0, np.cumsum((weights * (1 - labels))[order])[ends]]
    recall = true_positive / positive
    false_positive_rate = false_positive / negative

    # AP adds precision for each newly recovered fraction of positives.
    predicted_positive = np.maximum(true_positive[1:] + false_positive[1:], 1e-30)
    # Example: descending labels [1, 0, 1, 0] -> AP = 1/2*1 + 1/2*2/3 = 5/6.
    ap = np.sum(np.diff(recall) * true_positive[1:] / predicted_positive)
    # The same four labels give AUROC=3/4 when their scores are distinct.
    auc = np.sum(np.diff(false_positive_rate) * (recall[1:] + recall[:-1]) / 2)
    # Example: label=1, score=0.9 contributes (0.9 - 1)^2 = 0.01.
    brier = np.sum(weights * (scores - labels) ** 2) / weights.sum()
    return dict(AP=ap, AUROC=auc, Brier=brier)

In [ ]:
# Scores are already descending; labels [1,0,1,0] give AP=5/6 and AUROC=3/4.
example_predictions = pd.DataFrame({
    "mother_id": ["A", "A", "B", "B"],
    "target": [1, 0, 1, 0],
    "score": [0.9, 0.8, 0.7, 0.1],
})
example_data = prepare_metrics(example_predictions)
print("unweighted metrics =", weighted_metrics(example_data))

# Drawing A twice and B zero times repeats both A windows together.
example_counts = np.array([2, 0])
print("row weights =", example_counts[example_data["mother_index"]].tolist())
print("resampled metrics =", weighted_metrics(example_data, example_counts))

### 6.2. 산모를 복원추출해 95% 구간을 구한다

시험 산모가 $M$명이라면 매번 $M$명을 복원추출한다. 한 번의 추출에서 정상 또는 비정상만 남으면 순위 지표를 계산할 수 없어 그 반복은 건너뛴다. 유효한 반복의 2.5%·97.5% 분위수를 신뢰구간으로 쓰고, 실제 사용한 반복 수도 저장한다. 반복 2,000회와 seed 42를 두 모델에 동일하게 적용한다.

In [ ]:
def resample_mothers(data, n_boot, seed):
    samples = {metric: [] for metric in ("AP", "AUROC", "Brier")}
    mother_count = len(data["mothers"])
    # Both selected arms use seed 42 on the same test mothers.
    rng = np.random.default_rng(seed)

    for _ in range(n_boot):
        # Example: drawing indices [0, 0, 2] gives mother counts [2, 0, 1].
        # Draw exactly M mother IDs when there are M test mothers.
        drawn = rng.integers(mother_count, size=mother_count)
        counts = np.bincount(drawn, minlength=mother_count)
        metrics = weighted_metrics(data, counts)
        # A draw with no positives or no negatives is not a valid
        # replicate for AP/AUROC/Brier in the original rule.
        if metrics is None:
            continue
        for metric, value in metrics.items():
            if np.isfinite(value):
                samples[metric].append(value)
    return samples

In [ ]:
def bootstrap_intervals(prediction, n_boot=N_BOOTSTRAP, seed=BOOTSTRAP_SEED):
    data = prepare_metrics(prediction)
    # Point estimate uses each test mother once; bootstrap samples
    # quantify uncertainty around this fixed selected model.
    point = weighted_metrics(data)
    if point is None:
        raise ValueError("시험 평가에는 정상·비정상 구간이 모두 필요합니다")
    samples = resample_mothers(data, n_boot, seed)

    rows = []
    for metric, value in point.items():
        low, high = np.nan, np.nan
        if samples[metric]:
            # Example: if all 2,000 draws are valid, use their 2.5th and
            # 97.5th percentiles as the bootstrap interval.
            low, high = np.quantile(samples[metric], [0.025, 0.975])
        rows.append(dict(
            metric=metric, point=value, low=low, high=high,
            valid_bootstrap=len(samples[metric]),
            # Both counts refer only to the test prediction rows.
            mothers=len(data["mothers"]), segments=len(prediction),
        ))
    return pd.DataFrame(rows)

In [ ]:
metric_tables = []
for arm, prediction in prediction_tables.items():
    if prediction.empty:
        continue
    # Validation rows are saved for later comparisons, but excluded from test CI.
    test_prediction = prediction.loc[prediction["split"].eq("test")]
    metrics = bootstrap_intervals(test_prediction)
    # Example: two selected arms × three metrics = six result rows.
    metrics["model"] = arm
    metrics["feature"] = selections[arm]["feature"]
    metric_tables.append(metrics)

In [ ]:
# One metric row contains its point estimate, interval, usable draws,
# test mother/window counts, selected model, and selected feature.
metric_columns = [
    "metric", "point", "low", "high", "valid_bootstrap",
    "mothers", "segments", "model", "feature",
]
metrics_table = pd.DataFrame(columns=metric_columns)
if metric_tables:
    metrics_table = pd.concat(metric_tables, ignore_index=True)
display(metrics_table)

## 7. 선택한 특징의 반응 곡선을 확인한다

각 모델이 선택한 특징의 학습 최솟값부터 최댓값까지 200개 점에서 예측한다. 가로축은 특징값, 세로축은 예측 비정상 확률이다. 두 계열이 다른 특징을 선택할 수 있으므로 그래프를 각각 그린다. 곡선 모양은 모델을 다시 선택하는 기준으로 사용하지 않는다.

In [ ]:
for arm, model in selected_models.items():
    feature = selections[arm]["feature"]
    # Example: train range [110, 170] yields 200 evenly spaced inputs in that range.
    grid = np.linspace(training[feature].min(), training[feature].max(), 200)
    curve_input = pd.DataFrame({feature: grid})
    # These 200 points are for display, not another fitting step.
    probabilities = model.predict_proba(curve_input)[:, 1]

    figure, axis = plt.subplots(figsize=(7, 4))
    axis.plot(grid, probabilities, label=arm)
    axis.set(xlabel=feature, ylabel="Predicted abnormality", title=f"Validation-selected: {arm}")
    axis.legend()
    figure.tight_layout()
    plt.show()

## 8. 후보·선택·예측·평가를 따로 저장한다

| 파일 | 내용 |
| --- | --- |
| `single_ranking.csv` | 성공한 후보들의 검증 AUROC·AP; 원래 후보 순서 |
| `single_selection.json` | 계열별 선택 특징과 선택에 사용한 검증 점수 |
| `pred_single_linear.csv` | 선택된 선형 모델의 검증·시험 예측 |
| `pred_single_spline.csv` | 선택된 spline 모델의 검증·시험 예측 |
| `03_single_metrics.csv` | 선택 모델의 시험 AP·AUROC·Brier와 산모 신뢰구간 |
| `03_issues.csv` | 실패한 후보·보류된 선택과 사유 |
| `03_status.json` | 완료·부분 완료·미산출 상태, 선택 기록과 설정 |

기존 실험의 결과 파일명과 열 순서를 유지한다. 입력·분할이 준비되지 않았거나 계열의 후보가 덜 완료되면 해당 결과를 빈 표와 상태로 갱신해 이전 실행 결과가 남지 않게 한다. 오류는 이 단계의 CSV에 모은다. 모델 객체는 이번 실행의 메모리에만 있고 모델 파일로 저장하지 않는다.

In [ ]:
issue_table = pd.DataFrame(issues, columns=["stage", "job", "reason"])
status_name = "complete"
# Complete = both families selected; partial = one family;
# unavailable = no usable split or no successful candidate.
if not ready or ranking.empty:
    status_name = "unavailable"
elif len(selections) != len(FAMILIES):
    status_name = "partial"

status = dict(
    status=status_name, selected=selections, input_error=input_error,
    # Example: 55/56 means one candidate failed and its family stays unselected.
    candidates_completed=len(ranking), candidates_expected=len(FAMILIES) * len(CAT28),
    model_seed=MODEL_SEED, bootstrap_seed=BOOTSTRAP_SEED, bootstrap=N_BOOTSTRAP,
)
if not issue_table.empty:
    print("issue count =", len(issue_table))

In [ ]:
RESULTS.mkdir(parents=True, exist_ok=True)
# Always rewrite these tables; an empty current result must not leave
# a successful CSV from a previous run in place.
ranking.to_csv(RESULTS / "single_ranking.csv", index=False)
metrics_table.to_csv(RESULTS / "03_single_metrics.csv", index=False)
issue_table.to_csv(RESULTS / "03_issues.csv", index=False)

# Example: the selected spline arm writes pred_single_spline.csv.
for arm, prediction in prediction_tables.items():
    prediction.to_csv(RESULTS / f"pred_{arm}.csv", index=False)

In [ ]:
# Selection is recorded separately so later experiments can reuse the chosen feature.
for filename, values in (
    ("single_selection.json", selections),
    ("03_status.json", status),
):
    (RESULTS / filename).write_text(
        json.dumps(values, ensure_ascii=False, indent=2), encoding="utf-8",
    )
print("status =", status["status"])
print("results directory =", RESULTS)

## 선택 결과의 범위를 읽는다

선택한 모델은 각 계열의 28개 후보 중 검증 AUROC가 가장 높았던 모델이다. 모든 가능한 단변량 모델의 상한을 뜻하지 않는다. 산모 bootstrap 신뢰구간은 고정된 모델과 이번 시험 집단에서의 불확실성을 나타내며, 재학습이나 특징 선택의 변동은 포함하지 않는다.

다음 조합 모델 실험은 이 두 모델과 같은 시험 구간에서 비교한다. 같은 산모의 반복 기록·쌍태아를 함께 분할했지만, 과거 연구에서 이미 관측한 산모인지에 대한 한계는 01과 동일하다.

### 반출용 제외·오류 집계

개별 ID·경로·오류 전문을 제외한 분류별 건수다.

In [ ]:
show_issue_counts(issue_table["reason"])